# Day 13–16 · OOP (Part 3 of 3) — Polymorphism & Abstraction

**Duration:** 50–60 Minutes

### Learning Outcomes
- Understand **polymorphism**: one interface, many behaviours.
- Achieve it via **duck typing**, **method overriding**, and **operator overloading**.
- Simulate **method overloading** with default / variable arguments.
- Distinguish **runtime** vs **compile-time** polymorphism in Python.
- Understand **abstraction** and hide unnecessary detail.
- Build **Abstract Base Classes** with `abc.ABC` and `@abstractmethod`.

## 1. What is Polymorphism?

**Polymorphism** (Greek: *"many forms"*) means different objects can respond to the **same**
method call in **their own** way. You use one common interface, and each object does the right
thing for its type.

```text
        speak()
       /   |    \
    Dog   Cat   Cow
   Woof  Meow   Moo
```

**Key Notes:**
- The caller does not need to know the exact type — just that it supports the method.
- Polymorphism often uses inheritance, but **does not require it** (see duck typing).

## 2. Ways to Achieve Polymorphism in Python

| Technique | Idea |
|---|---|
| **Duck typing** | If it has the method, just call it — type doesn't matter |
| **Method overriding** | Subclass redefines a parent method (runtime) |
| **Operator overloading** | Give operators (`+`, `==`, …) custom meaning via dunder methods |
| **Method overloading (simulated)** | One method handles different argument counts via defaults / `*args` |

Python's polymorphism is mostly **runtime** — the method to run is decided while the program runs.

## 3. Duck Typing (Pure Pythonic Polymorphism)

> "If it walks like a duck and quacks like a duck, it must be a duck."

Python does not check an object's **type** — only whether it **has the method** being called.
Unrelated classes work together as long as they share the method name.

In [1]:
class Duck:
    def fly(self):
        print("Duck flying")

class Airplane:
    def fly(self):
        print("Airplane flying")

def lift_off(entity):        # doesn't care about the type...
    entity.fly()             # ...only that it has .fly()

lift_off(Duck())        # Duck flying
lift_off(Airplane())    # Airplane flying

Duck flying
Airplane flying


## 4. Method Overriding (Runtime Polymorphism)

The most common form: subclasses override a shared method, and the **object's own** version runs.
A single loop can drive many different types uniformly.

In [2]:
class Animal:
    def __init__(self, name):
        self.name = name
    def speak(self):
        return "Some sound"

class Dog(Animal):
    def speak(self):
        return f"{self.name} says Woof!"

class Cat(Animal):
    def speak(self):
        return f"{self.name} says Meow!"

# ONE interface (speak) drives MANY types
animals = [Dog("Buddy"), Cat("Whiskers"), Animal("Creature")]
for a in animals:
    print(a.speak())        # each object responds in its own way

Buddy says Woof!
Whiskers says Meow!
Some sound


## 5. Operator Overloading

Give operators custom meaning for your objects by defining dunder methods:
`__add__` (`+`), `__sub__` (`-`), `__mul__` (`*`), `__eq__` (`==`), `__str__` (`print`).

In [3]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):                 # p1 + p2
        return Point(self.x + other.x, self.y + other.y)

    def __eq__(self, other):                  # p1 == p2
        return self.x == other.x and self.y == other.y

    def __str__(self):
        return f"({self.x}, {self.y})"

p1 = Point(1, 2)
p2 = Point(3, 4)
print(p1 + p2)            # (4, 6)   -> __add__
print(p1 == Point(1, 2)) # True     -> __eq__

(4, 6)
True


## 6. Method Overloading (Simulated)

Languages like C++/Java allow **many methods with the same name** but different parameters. Python
does **not** — a later definition simply replaces the earlier one. Instead we simulate it with
**default arguments** or **`*args` / `**kwargs`**.

In [4]:
# Using default arguments: one method, different numbers of inputs
class MathOps:
    def add(self, a, b=0, c=0):
        return a + b + c

m = MathOps()
print(m.add(5))          # 5
print(m.add(5, 10))      # 15
print(m.add(5, 10, 15))  # 30

5
15
30


In [5]:
# Using *args / **kwargs for fully flexible arguments
def display_info(*args, **kwargs):
    print("Positional:", args)
    print("Keyword   :", kwargs)

display_info(1, 2, 3, name="Alice", age=30)

Positional: (1, 2, 3)
Keyword   : {'name': 'Alice', 'age': 30}


## 7. Runtime vs Compile-time Polymorphism

| Feature | Supported in Python? | Notes |
|---|---|---|
| Compile-time polymorphism | 🚫 No (real) | Python is interpreted & dynamically typed |
| **Operator overloading** | ✅ Simulated | Via magic methods (`__add__`, `__mul__`, …) |
| **Method overloading** | ✅ Simulated | Use default / `*args` / `**kwargs` instead |
| **Method overriding** | ✅ Yes (real, runtime) | Inheritance-based + duck typing |

**Key Notes:**
- "Real" polymorphism in Python is **runtime** (overriding, duck typing).
- Overloading is **simulated**, not a true language feature.

## 8. One Function, Many Types

Because of polymorphism, a single function can process a whole list of different objects — as long
as they all support the expected method.

In [6]:
class Circle:
    def __init__(self, r): self.r = r
    def area(self): return round(3.14159 * self.r ** 2, 2)

class Square:
    def __init__(self, s): self.s = s
    def area(self): return self.s ** 2

def total_area(shapes):
    return sum(shape.area() for shape in shapes)   # same .area() call

print(total_area([Circle(5), Square(4), Circle(2)]))

107.11000000000001


## 9. What is Abstraction?

**Abstraction** means exposing only the **essential** features of something and **hiding the
complex details**. You care about *what* an object does, not *how* it does it.

**Real-life analogy:** you drive a car with a steering wheel and pedals (the interface) without
knowing how the engine works internally (the implementation).

**Why abstraction?**
- **Standardises** an interface every subclass must follow.
- **Simplifies** usage — callers see a clean, minimal surface.
- **Reduces mistakes** — subclasses are forced to implement required methods.

## 10. Abstract Base Classes (ABC)

In Python, abstraction is done with **Abstract Base Classes** from the **`abc`** module.

- An **abstract class** cannot be instantiated directly; it is meant to be **subclassed**.
- An **abstract method** (marked `@abstractmethod`) has no body and **must** be implemented by
  every concrete subclass.

**Syntax**
```python
from abc import ABC, abstractmethod

class Base(ABC):
    @abstractmethod
    def do_something(self):
        pass
```

In [7]:
from abc import ABC, abstractmethod

class Shape(ABC):                     # abstract base class
    @abstractmethod
    def area(self):                   # abstract method - no body
        pass

    @abstractmethod
    def perimeter(self):
        pass

# You CANNOT create an object of an abstract class:
try:
    s = Shape()
except TypeError as e:
    print("cannot instantiate abstract class:", e)

cannot instantiate abstract class: Can't instantiate abstract class Shape without an implementation for abstract methods 'area', 'perimeter'


## 11. Concrete Subclasses Must Implement Every Abstract Method

A subclass becomes usable only when it provides **all** the abstract methods. Miss one and it
stays abstract (and can't be instantiated).

In [8]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self): pass
    @abstractmethod
    def perimeter(self): pass

class Rectangle(Shape):               # concrete: implements BOTH
    def __init__(self, w, h):
        self.w = w
        self.h = h
    def area(self):
        return self.w * self.h
    def perimeter(self):
        return 2 * (self.w + self.h)

r = Rectangle(4, 3)
print("area     :", r.area())         # 12
print("perimeter:", r.perimeter())    # 14

area     : 12
perimeter: 14


In [9]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self): pass
    @abstractmethod
    def perimeter(self): pass

# Missing perimeter() -> still abstract -> cannot instantiate
class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self):
        return round(3.14159 * self.r ** 2, 2)

try:
    c = Circle(5)
except TypeError as e:
    print("still abstract:", e)

still abstract: Can't instantiate abstract class Circle without an implementation for abstract method 'perimeter'


## 12. Abstract Classes Can Also Have Normal Methods

An abstract class may mix **abstract methods** (must override) with **concrete methods** (shared,
ready-to-use). This is the reference `Vehicle` example.

In [10]:
from abc import ABC, abstractmethod

class Vehicle(ABC):
    def __init__(self, brand):
        self.brand = brand

    @abstractmethod
    def start(self):                  # each vehicle starts differently
        pass

    def show_brand(self):             # shared concrete method
        print(f"Brand: {self.brand}")

class Car(Vehicle):
    def start(self):
        print("Car started with key ignition.")

class ElectricScooter(Vehicle):
    def start(self):
        print("Electric Scooter started with a button.")

for v in [Car("Toyota"), ElectricScooter("Ather")]:
    v.show_brand()      # shared behaviour
    v.start()           # each type's own behaviour

Brand: Toyota
Car started with key ignition.
Brand: Ather
Electric Scooter started with a button.


## 13. Abstraction vs Encapsulation

They sound similar but solve different problems.

| | Abstraction | Encapsulation |
|---|---|---|
| Hides | **complexity** (how it works) | **data** (internal state) |
| Focus | design — *what* an object does | implementation — *protecting* data |
| Achieved with | abstract classes / methods (`abc`) | access modifiers, `@property` |
| Goal | a clean, required interface | safe, validated data access |

## 14. Common Mistakes

**1. Trying to instantiate an abstract class** — raises `TypeError`; make a concrete subclass.

**2. Forgetting to implement an abstract method** — the subclass stays abstract and cannot be
instantiated.

**3. Expecting real method overloading** — a second `def` with the same name **replaces** the
first. Use default / `*args` arguments instead.

**4. Overusing operator overloading** — only overload operators when the meaning is obvious
(`Vector + Vector` is clear; `Person + Person` is not).

## 15. Mini Example — Shapes (Abstraction + Polymorphism Together)

An abstract `Shape` interface with several concrete shapes, all used through the **same** methods.

In [11]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self): pass
    def describe(self):                       # shared concrete method
        return f"{self.__class__.__name__} with area {self.area()}"

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return round(3.14159 * self.r ** 2, 2)

class Rectangle(Shape):
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h

class Triangle(Shape):
    def __init__(self, b, h): self.b, self.h = b, h
    def area(self): return 0.5 * self.b * self.h

# Polymorphism: one loop, many shapes
for shape in [Circle(5), Rectangle(4, 3), Triangle(6, 8)]:
    print(shape.describe())

Circle with area 78.54
Rectangle with area 12
Triangle with area 24.0


## 16. Summary

- **Polymorphism** = one interface, many behaviours.
- Achieve it with **duck typing**, **method overriding**, and **operator overloading**.
- Python has **no true method overloading** — simulate with default / `*args` arguments.
- Python's real polymorphism is **runtime** (overriding + duck typing).
- **Abstraction** hides complexity behind a clean interface using **`abc.ABC`** and
  **`@abstractmethod`**.
- Abstract classes **can't be instantiated**; subclasses **must** implement all abstract methods.
- **Abstraction** hides *complexity*; **encapsulation** hides *data*.

## 17. Practice Questions

1. Create `Dog`, `Cat`, `Cow` classes each with a `speak()` method and call them in one loop.
2. Demonstrate duck typing with two unrelated classes that both have a `.start()` method.
3. Override a `describe()` method in two subclasses of a common parent.
4. Overload `+` for a `Money` class using `__add__`.
5. Overload `==` for a `Point` class using `__eq__`.
6. Simulate method overloading with a `volume(l, b=1, h=1)` method.
7. Use `*args` to write a `maximum(*nums)` function that works for any count of numbers.
8. Create an abstract class `Payment` with an abstract `pay(amount)` method.
9. Implement `CreditCard` and `UPI` subclasses of `Payment`.
10. Show that instantiating `Payment` directly raises a `TypeError`.